# Proyecto Integrador 2 · Parte III — Recomendador por contenido
## Tareas 15 y 16: TF-IDF por proveedor y búsqueda con HNSW

**Pregunta de interés P2:** *¿Qué proveedores son sustitutos entre sí para una misma entidad o categoría, y qué afinidades entidad–proveedor son inusuales?*

**Idea del recomendador por contenido:** dos proveedores se parecen si **venden cosas parecidas**. Entonces, a una entidad se le recomiendan proveedores parecidos a los que ya le adjudicó, **aunque nunca les haya comprado**.

**Qué hace este notebook**
1. Arma un **texto por proveedor** con los procesos que ganó (solo antes de la fecha de corte: nunca se mira el test).
2. Lo convierte en vector con **TF-IDF + proyección aleatoria a 512** (lo mismo del Proyecto 1) y lo indexa con **HNSW**.
3. El **perfil de una entidad** es el promedio de los vectores de sus proveedores. Se recomiendan los 50 más cercanos que la entidad no tenga.

**Entrada:** Parquet del Proyecto 1 y de los notebooks 1. **Salida:** `rec_contenido` (Parquet), los vectores para el notebook 6 y `contenido_ejemplos.json` para la página web.

## 0. Configuración

In [1]:
import os, json, time
import numpy as np
import pandas as pd
import faiss
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.random_projection import SparseRandomProjection
from sklearn.preprocessing import normalize

RUTA_PARQUET = os.environ.get("P2_PARQUET", "/content/drive/MyDrive/proyecto_integrador_seace/parquet")
RUTA_P2      = os.environ.get("P2_DATOS",   "/content/drive/MyDrive/proyecto_integrador_seace/p2")
WEB          = os.environ.get("P2_WEB",     "/content/drive/MyDrive/proyecto_integrador_seace/web_artefactos/p2")
os.makedirs(WEB, exist_ok=True)

FECHA_CORTE = "2026-03-01"   # el mismo del notebook 1
MAX_TEXTOS  = 30             # por proveedor se usan sus 30 procesos ganados más recientes
DIM         = 512            # dimensiones del vector (igual que en el Proyecto 1)
K_REC       = 50             # recomendaciones que se guardan por entidad
M, EF_CONSTRUCCION, EF_BUSQUEDA = 32, 100, 64   # HNSW (mismos M y efConstruction del Proyecto 1)
SEED = 42

## 1. Texto de cada proveedor (solo con datos de train)
Para cada proveedor del universo se juntan los textos (`texto`: título y descripción del proceso, ya normalizado en el Proyecto 1) de **los procesos que ganó antes de `FECHA_CORTE`**.

Se usan los procesos ganados y no los ítems porque la tabla de ítems no dice a qué proveedor se adjudicó cada ítem cuando un proceso tiene varios ganadores.

In [2]:
train = pd.read_parquet(f"{RUTA_P2}/interacciones_train")
test  = pd.read_parquet(f"{RUTA_P2}/interacciones_test")
dim_prov = pd.read_parquet(f"{RUTA_P2}/proveedor_dim")
dim_ent  = pd.read_parquet(f"{RUTA_P2}/entidad_dim")

def nombre_visible(ruc, nombre):   # datos personales: persona natural → RUC enmascarado en vez de nombre
    r = str(ruc); return f"Persona natural (RUC {r[:5]}****{r[-2:]})" if r[:2] in ("10", "15", "16", "17") else nombre
dim_web = dim_prov.assign(nombre=[nombre_visible(r, n) for r, n in zip(dim_prov.ruc, dim_prov.nombre)])

prov_ids = np.sort(train.proveedor_idx.unique())      # universo de proveedores (en el orden de las columnas)
ent_ids  = np.sort(train.entidad_idx.unique())        # universo de entidades (en el orden de las filas)
print("proveedores:", len(prov_ids), "| entidades:", len(ent_ids))

adj  = pd.read_parquet(f"{RUTA_PARQUET}/adjudicaciones", columns=["release_id", "proveedor_ruc"])
proc = pd.read_parquet(f"{RUTA_PARQUET}/procesos", columns=["release_id", "fecha_pub", "texto"])
proc = proc[proc.fecha_pub < FECHA_CORTE]

t = (adj.merge(dim_prov[["proveedor_idx", "ruc"]], left_on="proveedor_ruc", right_on="ruc")
        .merge(proc, on="release_id"))
t = t[t.proveedor_idx.isin(prov_ids)].drop_duplicates(["proveedor_idx", "release_id"])
t = t.sort_values("fecha_pub", ascending=False).groupby("proveedor_idx").head(MAX_TEXTOS)

textos = t.groupby("proveedor_idx").texto.agg(" ".join).reindex(prov_ids)
print("proveedores sin texto:", textos.isna().sum(), "| textos por proveedor (mediana):", t.groupby("proveedor_idx").size().median())
print("\nEjemplo:", dim_web.set_index("proveedor_idx").nombre[prov_ids[0]], "→", textos.iloc[0][:150])

proveedores: 9833 | entidades: 2331


proveedores sin texto: 0 | textos por proveedor (mediana): 4.0

Ejemplo: Persona natural (RUC 10000****50) → servicio de revision reparacion y mantenimiento de la maquina autoclave marca bmt area de central de esterilizacion contratacion servicio de mantenimi


## 2. Vectores: TF-IDF y proyección aleatoria
- **TF-IDF:** cada palabra (y cada par de palabras) pesa más si aparece mucho en ese texto y poco en los demás. "de" casi no pesa; "luminarias" pesa mucho.
- **Proyección aleatoria a 512:** reduce el vector largo a 512 números y casi conserva las distancias. Después se normaliza (largo 1), así el **producto punto es la similitud coseno**.

In [3]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=100_000, sublinear_tf=True)
X = tfidf.fit_transform(textos.fillna("")).tocsr()
V = normalize(SparseRandomProjection(n_components=DIM, dense_output=True, random_state=SEED).fit_transform(X)).astype("float32")
print("TF-IDF:", X.shape, "| vectores:", V.shape)

# ¿Cuánto se pierde al reducir? Vecinos con TF-IDF completo frente a los de 512 números (300 proveedores al azar)
q = np.random.RandomState(SEED).choice(len(prov_ids), 300, replace=False)
sim_full = (X[q] @ X.T).toarray(); sim_full[np.arange(300), q] = -1
sim_red  = V[q] @ V.T;             sim_red[np.arange(300), q] = -1
v_full = np.argsort(-sim_full, axis=1)[:, :10]; v_red = np.argsort(-sim_red, axis=1)[:, :10]
solape = np.mean([len(set(a) & set(b)) / 10 for a, b in zip(v_full, v_red)])
print(f"de los 10 vecinos según TF-IDF completo, {100*solape:.0f}% están también en los de 512 números")

TF-IDF: (9833, 61120) | vectores: (9833, 512)


de los 10 vecinos según TF-IDF completo, 62% están también en los de 512 números


## 3. Índice HNSW
**HNSW** guarda cada vector conectado con sus vecinos cercanos, en una red de varias capas; la búsqueda "baja" por esa red en vez de comparar con todos. Parámetros (los mismos `M` y `efConstruction` del Proyecto 1, sin ajustar):
- `M`: conexiones por vector (32).
- `efConstruction`: candidatos que se revisan al construir el índice (100).
- `efSearch`: candidatos que se revisan al buscar (64).

Con solo ~10 mil vectores la búsqueda exacta también es instantánea. Se usa HNSW por continuidad con el Proyecto 1 (donde sí hacía falta, con 163,208 textos) y se **verifica su recall** contra la búsqueda exacta.

In [4]:
index = faiss.IndexHNSWFlat(DIM, M, faiss.METRIC_INNER_PRODUCT)
index.hnsw.efConstruction = EF_CONSTRUCCION
index.add(V)

exacto = faiss.IndexFlatIP(DIM); exacto.add(V)
index.hnsw.efSearch = EF_BUSQUEDA
_, ia = index.search(V[q], 11); _, ie = exacto.search(V[q], 11)
recall = np.mean([len(set(a[1:]) & set(b[1:])) / 10 for a, b in zip(ia, ie)])   # [1:] quita al propio proveedor
print(f"recall@10 de HNSW frente a la búsqueda exacta (efSearch={EF_BUSQUEDA}): {100*recall:.1f}%")

recall@10 de HNSW frente a la búsqueda exacta (efSearch=64): 98.8%


## 4. Recomendar a cada entidad
- **Perfil de la entidad:** promedio de los vectores de los proveedores a los que ya adjudicó (en train), normalizado.
- Se buscan los más cercanos en el índice y se **quitan los que la entidad ya tiene**. Como una entidad puede tener muchos proveedores, se piden `K_REC` + los que ya tiene.

In [5]:
pos_ent  = {e: i for i, e in enumerate(ent_ids)}
pos_prov = {p: i for i, p in enumerate(prov_ids)}
R = csr_matrix((np.ones(len(train), dtype="float32"),
                (train.entidad_idx.map(pos_ent).values, train.proveedor_idx.map(pos_prov).values)),
               shape=(len(ent_ids), len(prov_ids)))
perfil = normalize(R @ V).astype("float32")

n_vistos = np.asarray(R.sum(1)).ravel().astype(int)
K_BUSCAR = int(min(len(prov_ids), K_REC + n_vistos.max()))
index.hnsw.efSearch = max(EF_BUSQUEDA, K_BUSCAR)
t0 = time.perf_counter()
S, I = index.search(perfil, K_BUSCAR)
print(f"búsqueda para {len(ent_ids)} entidades (k={K_BUSCAR}): {time.perf_counter()-t0:.1f} s")

filas = []
for i, e in enumerate(ent_ids):
    vistos = set(R.indices[R.indptr[i]:R.indptr[i+1]])
    r = 0
    for j, s in zip(I[i], S[i]):
        if j < 0 or j in vistos:
            continue
        r += 1
        filas.append((e, prov_ids[j], float(s), r))
        if r == K_REC:
            break
rec = pd.DataFrame(filas, columns=["entidad_idx", "proveedor_idx", "score", "rank"])
rec.to_parquet(f"{RUTA_P2}/rec_contenido.parquet", index=False)
print("rec_contenido:", len(rec), "filas |", rec.entidad_idx.nunique(), "entidades | proveedores distintos recomendados:", rec.proveedor_idx.nunique())

# vectores y palabras clave para el notebook 6
np.save(f"{RUTA_P2}/vectores_proveedor.npy", V); np.save(f"{RUTA_P2}/vectores_proveedor_ids.npy", prov_ids)
terminos = np.array(tfidf.get_feature_names_out())
def clave(i, n=5):
    fila = X[i]; return ", ".join(terminos[fila.indices[np.argsort(-fila.data)[:n]]])
pd.DataFrame({"proveedor_idx": prov_ids, "palabras_clave": [clave(i) for i in range(len(prov_ids))]}).to_parquet(f"{RUTA_P2}/proveedores_palabras.parquet", index=False)

búsqueda para 2331 entidades (k=751): 1.7 s


rec_contenido: 116550 filas | 2331 entidades | proveedores distintos recomendados: 7434


## 5. Ejemplos para la página web

In [6]:
nombre_prov = dim_web.set_index("proveedor_idx").nombre
info_ent = dim_ent.set_index("entidad_idx")
palabras = pd.read_parquet(f"{RUTA_P2}/proveedores_palabras.parquet").set_index("proveedor_idx").palabras_clave

# 12 entidades al azar, con entre 5 y 60 proveedores en train (para que el ejemplo se lea)
cand = [e for e, n in zip(ent_ids, n_vistos) if 5 <= n <= 60]
elegidas = np.random.RandomState(SEED).choice(cand, 12, replace=False)
ejemplos = []
for e in elegidas:
    tr = train[train.entidad_idx == e].sort_values("n_adjudicaciones", ascending=False).head(5)
    rr = rec[rec.entidad_idx == e].head(10)
    ejemplos.append({
        "entidad": info_ent.loc[e, "nombre"], "tipo": info_ent.loc[e, "entidad_tipo"], "departamento": info_ent.loc[e, "entidad_departamento"],
        "proveedores_actuales": [{"nombre": nombre_prov[p], "n_adjudicaciones": int(n), "palabras_clave": palabras[p]} for p, n in zip(tr.proveedor_idx, tr.n_adjudicaciones)],
        "recomendados": [{"nombre": nombre_prov[p], "score": round(float(s), 3), "palabras_clave": palabras[p]} for p, s in zip(rr.proveedor_idx, rr.score)],
    })

meta = {"proveedores_indexados": int(len(prov_ids)), "entidades": int(len(ent_ids)), "dim": DIM,
        "hnsw": {"M": M, "efConstruction": EF_CONSTRUCCION, "efSearch": EF_BUSQUEDA, "recall_10_vs_exacta": round(float(recall), 4)},
        "solape_tfidf_vs_512": round(float(solape), 3), "textos_por_proveedor_max": MAX_TEXTOS, "fecha_corte": FECHA_CORTE}
with open(f"{WEB}/contenido_ejemplos.json", "w", encoding="utf-8") as f:
    json.dump({"meta": meta, "ejemplos": ejemplos}, f, ensure_ascii=False, indent=1)
print("guardado: contenido_ejemplos.json")
e0 = ejemplos[0]; print(e0["entidad"], "| tiene:", [p["nombre"] for p in e0["proveedores_actuales"][:3]]); print("recomendado:", [(p["nombre"], p["score"]) for p in e0["recomendados"][:3]])

guardado: contenido_ejemplos.json
GOBIERNO REGIONAL DE PUNO- HOSPITAL REGIONAL MANUEL NUÑEZ BUTRON | tiene: ['DIAGNOSTICA PERUANA S.A.C.', 'SISTEMAS ANALITICOS SRL', 'AIR MAJORO SOCIEDAD ANONIMA']
recomendado: [('MEDIFARMA S A', 0.396), ('PHARMA HOSTING PERU SAC', 0.39), ('UNILENE S.A.C.', 0.383)]
